In [ ]:
import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

In [ ]:
df = pd.read_excel("../Data/Social_Media_Posting_Report_ANONYMIZED.xlsx")

q5_col = [c for c in df.columns if c.startswith("Q5")][0]
df = df.rename(columns={q5_col: "Q5_caption"})

def clean_text(x):
    if pd.isna(x): return ""
    s = str(x).strip()
    return "" if s.upper() == "NA" else s

df["content_text"] = df["Q5_caption"].apply(clean_text)
df["has_text"] = df["content_text"].str.len() > 0
print(f"Usable rows: {df['has_text'].sum()} / {len(df)}")

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from tqdm import tqdm
import gc

FOUNDATIONS = ["care", "fairness", "loyalty", "authority", "sanctity"]
MODEL_BASE = "joshnguyen/mformer-"

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Shared tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_BASE + FOUNDATIONS[0])

# Prepare texts
texts = df.loc[df["has_text"], "content_text"].tolist()
texts = [t[:1500] for t in texts]
print(f"Will process {len(texts)} posts across {len(FOUNDATIONS)} foundations")

# Process ONE foundation at a time
morality_results = {}

for foundation in FOUNDATIONS:
    print(f"\n--- Loading {foundation} model ---")
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_BASE + foundation
    ).to(device).eval()
    
    scores = []
    batch_size = 16
    
    for i in tqdm(range(0, len(texts), batch_size), desc=f"{foundation}"):
        batch = texts[i:i+batch_size]
        inputs = tokenizer(
            batch, padding=True, truncation=True, return_tensors="pt"
        ).to(device)
        
        with torch.no_grad():
            outputs = model(**inputs)
        probs = torch.softmax(outputs.logits, dim=1)[:, 1].cpu().tolist()
        scores.extend(probs)
    
    morality_results[foundation] = scores
    
    # Free memory before loading the next model
    del model
    gc.collect()
    if device == "cuda":
        torch.cuda.empty_cache()
    
    print(f"  {foundation} done: {len(scores)} scores")

print("\n✅ All 5 foundations complete.")

In [ ]:
mor_cols = [f"mor_{f}" for f in FOUNDATIONS]

# Drop existing morality columns if they exist (safe re-run)
df = df.drop(columns=[c for c in mor_cols + ["dominant_foundation", "dominant_foundation_score"] if c in df.columns])

# Build morality dataframe
mor_df = pd.DataFrame({f"mor_{f}": morality_results[f] for f in FOUNDATIONS})
mor_df.index = df.index[df["has_text"]]
df = df.join(mor_df)

# Compute dominant foundation only for posts with text
df["dominant_foundation"] = None
df["dominant_foundation_score"] = None
df.loc[df["has_text"], "dominant_foundation"] = (
    df.loc[df["has_text"], mor_cols].idxmax(axis=1).str.replace("mor_", "")
)
df.loc[df["has_text"], "dominant_foundation_score"] = df.loc[df["has_text"], mor_cols].max(axis=1)

print(df[["student_id", "dominant_foundation", "dominant_foundation_score"]].head())

In [ ]:
print("=== Dominant foundation counts ===")
print(df["dominant_foundation"].value_counts(dropna=False))
print()
print("=== Mean scores across all posts ===")
print(df[mor_cols].mean().sort_values(ascending=False))
print()
print("=== Per-student dominant foundation breakdown ===")
print(pd.crosstab(df["student_id"], df["dominant_foundation"]))

In [ ]:
for f in FOUNDATIONS:
    subset = df[df["dominant_foundation"] == f]
    print(f"\n=== {f.upper()} ({len(subset)} posts) ===")
    if len(subset) == 0: continue
    top = subset.nlargest(3, f"mor_{f}")
    for _, r in top.iterrows():
        print(f"  [{r[f'mor_{f}']:.2f}] {r['content_text'][:180]}")

In [ ]:
df.to_csv("../Outputs/morality_output_709_anonymized.csv", index=False)
print("Saved to ../Outputs/morality_output_709_anonymized.csv")

# Morality Inference — Conclusion

## What we found

The model ran on 703 of 709 posts using MFormer's 5 separate classifiers for the moral foundations: care, fairness, loyalty, authority, and sanctity. Each post got an independent probability score for each foundation, rather than a single label.

The dominant foundation breakdown shows three near-equal categories at the top: 28% sanctity, 27% authority, 26% care. Fairness (11%) and loyalty (7%) are much smaller categories.

An important caveat about the numbers: the raw scores are low. The highest mean across all posts is care at 0.13, with the others sitting around 0.05 to 0.08. This means most posts don't strongly engage any particular foundation. The "dominant foundation" label tells us which one a post is most related to, not that the post is strongly about that foundation. A post can be labeled "sanctity" while still only mildly engaging sanctity themes.

The most interesting finding, again, is at the student level. Students show real differences in moral content exposure. S05's feed is care-heavy with 28 care posts. S07 stands out with 18 fairness posts, about 3x the average. S06 has zero sanctity posts in their entire feed, while S03 and S09 have 24 and 27 respectively. These per-student differences are likely the project's strongest signal.

## Where the model works well

Care detection is the cleanest of the five. The top examples are unambiguously care content: prayer posts wishing someone gets a job, a baby macaque adopted after maternal abandonment, someone returning to content creation after a miscarriage. The model correctly identifies posts that center on compassion, harm, or care for others.

Fairness detection is also solid. The top examples (sexual health education scholarship, women winning in athletics, a careful performance representing disability) share a clear thread of equity, access, and representation. The model is picking up real fairness content.

## Where the model struggles

Authority detection is mixed. Genuine authority content (a JFK conspiracy post, an airline headphone policy update) is correctly identified, but the model also flags nostalgic family content like "I'll tell my kids about catching trains" as authority, probably from family/tradition keywords. About one in three top authority posts looks like a false positive.

Sanctity is the noisiest category. It catches genuine purity content like a Taoist temple incense ritual, but also fires on anything mentioning disgust or grossness ironically. A joke post saying "her greed sickens me #goldenretriever" was scored as 99% sanctity because of the word "sickens." This keyword reactivity inflates the sanctity count.

Loyalty detection is the weakest. Only 51 posts end up labeled as loyalty, and even the top-scoring ones are weak matches. A Chadwick Boseman tribute makes sense, but "a PowerPoint every dog can stand behind" is clearly a pun-based false positive on the word "behind." Social media feeds don't seem to contain much explicit loyalty content, which itself is a finding.

## Bottom line

The pipeline runs cleanly. Care and fairness are reliable. Authority and sanctity carry meaningful noise. Loyalty is sparse and weak.

The biggest methodological point: the absolute foundation scores are low across the board, so the dominant_foundation label on a single post shouldn't be taken too seriously. The signal lives in the relative differences between students, where the variance is real and large.